# ACE A11 PoC v2 (Grad-CAM-guided regions)

This notebook runs `src/ace_pipeline_v2.py` end-to-end:
1. Grad-CAM spoof map per utterance
2. Connected-component region proposals
3. Masked patch embedding from layer3
4. Per-scale clustering + spoof-vs-bonafide scoring
5. Export concept folders for TCAV curation

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

CANDIDATES = [
    Path('/home/SpeakerRec/BioVoice'),
    Path.cwd().resolve(),
    Path.cwd().resolve().parents[1] if len(Path.cwd().resolve().parents) > 1 else Path.cwd().resolve(),
]
ROOT = None
for c in CANDIDATES:
    if (c / 'resnet_293' / 'ace' / 'ace_a11_poc' / 'src' / 'ace_pipeline_v2.py').exists():
        ROOT = c
        break
if ROOT is None:
    raise RuntimeError('Could not locate BioVoice root. Set ROOT manually.')

ACE_DIR = ROOT / 'resnet_293' / 'ace' / 'ace_a11_poc'
SRC_DIR = ACE_DIR / 'src'
CFG_PATH = ACE_DIR / 'configs' / 'a11_ace_v2_config.yaml'

sys.path.insert(0, str(SRC_DIR))
from ace_pipeline_v2 import run_ace_v2  # noqa: E402

print('ROOT =', ROOT)
print('ACE_DIR =', ACE_DIR)
print('CFG_PATH =', CFG_PATH)
print('Config exists =', CFG_PATH.exists())

In [ ]:
# Run v2 pipeline
meta = run_ace_v2(CFG_PATH)
print(json.dumps(meta, indent=2))

In [ ]:
# Inspect artifacts
import pandas as pd
out_dir = Path(meta['output_dir'])
score_csv = out_dir / 'cluster_scores.csv'
concepts_dir = out_dir / 'concepts_auto'
print('Output dir:', out_dir)
print('cluster_scores.csv exists:', score_csv.exists())
print('concepts_auto exists:', concepts_dir.exists())
if score_csv.exists():
    df = pd.read_csv(score_csv)
    display(df.head(20))
if concepts_dir.exists():
    cdirs = sorted([p for p in concepts_dir.iterdir() if p.is_dir()])
    print('num cluster dirs:', len(cdirs))
    for p in cdirs[:10]:
        print(p.name, 'npy=', len(list(p.glob('*.npy'))))